<a href="https://colab.research.google.com/github/IrisCheon/NLP-practice/blob/main/Stance_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#■ Stance Detection

This notebook explores stance detection using TF-IDF vectorisation and Logistic Regression on the SemEval-2016 Twitter stance dataset.

The analysis includes:
- exploratory data analysis (EDA)
- multi-class stance classification
- target + tweet vs tweet-only comparison
- confusion matrix analysis
- class imbalance observations
- feature importance and mean TF-IDF inspection
- error analysis for ambiguous and low-information tweets

The goal of this notebook is not only to evaluate classification performance, but also to explore how stance detection differs from standard sentiment classification.

In [100]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [101]:
!pip install datasets
from datasets import load_dataset

dataset = load_dataset("krishnagarg09/SemEval2016Task6")

In [102]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Tweet', 'Target', 'Stance'],
        num_rows: 2160
    })
    validation: Dataset({
        features: ['Tweet', 'Target', 'Stance'],
        num_rows: 359
    })
    test: Dataset({
        features: ['Tweet', 'Target', 'Stance'],
        num_rows: 1080
    })
})

In [103]:
dataset["train"][0]

{'Tweet': 'dear lord thank u for all of ur blessings forgive my sins lord give me strength and energy for this busy day ahead #blessed #hope #SemST',
 'Target': 'Atheism',
 'Stance': 'AGAINST'}

In [104]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

#■ 데이터 확인

In [105]:
train_df.head()

,Tweet,Target,Stance
0,dear lord thank u for all of ur blessings forg...,Atheism,AGAINST
1,"Blessed are the peacemakers, for they shall be...",Atheism,AGAINST
2,I am not conformed to this world. I am transfo...,Atheism,AGAINST
3,Salah should be prayed with #focus and #unders...,Atheism,AGAINST
4,"Papa God, i pray that You shower me with more ...",Atheism,AGAINST


In [106]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2160 entries, 0 to 2159
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Tweet   2160 non-null   object
 1   Target  2160 non-null   object
 2   Stance  2160 non-null   object
dtypes: object(3)
memory usage: 50.8+ KB


In [107]:
train_df.columns

Index(['Tweet', 'Target', 'Stance'], dtype='object')

In [108]:
train_df["Target"].value_counts()

,count
Target,
Hillary Clinton,591
Feminist Movement,569
Legalization of Abortion,561
Atheism,439


In [109]:
train_df["Stance"].value_counts()

,count
Stance,
AGAINST,1143
NONE,534
FAVOR,483


In [110]:
train_df["input_text"] = train_df["Target"] + "[SEP]" + train_df["Tweet"]
test_df["input_text"] = test_df["Target"] + "[SEP]" + test_df["Tweet"]

test_df.head()

,Tweet,Target,Stance,input_text
0,He who exalts himself shall be humbled; a...,Atheism,AGAINST,Atheism[SEP]He who exalts himself shall b...
1,RT @prayerbullets: I remove Nehushtan -previou...,Atheism,AGAINST,Atheism[SEP]RT @prayerbullets: I remove Nehush...
2,@Brainman365 @heidtjj @BenjaminLives I have so...,Atheism,AGAINST,Atheism[SEP]@Brainman365 @heidtjj @BenjaminLiv...
3,#God is utterly powerless without Human interv...,Atheism,AGAINST,Atheism[SEP]#God is utterly powerless without ...
4,@David_Cameron Miracles of #Multiculturalism...,Atheism,AGAINST,Atheism[SEP]@David_Cameron Miracles of #Mult...


In [111]:
X_train = train_df["input_text"]
y_train = train_df["Stance"]

X_test = test_df["input_text"]
y_test = test_df["Stance"]

# ■ ML

In [112]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [113]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features = 10000,
        ngram_range = (1,2),
        stop_words = "english"
    )),
    ("clf", LogisticRegression(
        max_iter = 1000
    ))
])

In [114]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf', LogisticRegression(max_iter=1000))])

In [115]:
y_pred = model.predict(X_test)

In [116]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
# 전반적으로 recall 상당히 떨어짐 > against는 아닌것도 against로 잡는 문제
# None의 recall이 극히 낮은것을 보면, None을 aginst로 잡는지?

              precision    recall  f1-score   support

     AGAINST       0.69      0.82      0.75       704
       FAVOR       0.31      0.31      0.31       181
        NONE       0.53      0.19      0.28       195

    accuracy                           0.62      1080
   macro avg       0.51      0.44      0.45      1080
weighted avg       0.60      0.62      0.59      1080



In [117]:
from sklearn.metrics import confusion_matrix

labels = ["AGAINST", "FAVOR", "NONE"]

cm = confusion_matrix(y_test, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"actual_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels]
)

cm_df

# 대부분 against로 판단하는 것으로 보임

,pred_AGAINST,pred_FAVOR,pred_NONE
actual_AGAINST,576,102,26
actual_FAVOR,117,56,8
actual_NONE,137,20,38


# ■ Results 보기

In [118]:
results = pd.DataFrame({
    "target" : test_df["Target"],
    "tweet" : test_df["Tweet"],
    "actual" : y_test,
    "predicted" : y_pred
})

results

,target,tweet,actual,predicted
0,Atheism,He who exalts himself shall be humbled; a...,AGAINST,AGAINST
1,Atheism,RT @prayerbullets: I remove Nehushtan -previou...,AGAINST,AGAINST
2,Atheism,@Brainman365 @heidtjj @BenjaminLives I have so...,AGAINST,AGAINST
3,Atheism,#God is utterly powerless without Human interv...,AGAINST,AGAINST
4,Atheism,@David_Cameron Miracles of #Multiculturalism...,AGAINST,AGAINST
...,...,...,...,...
1075,Legalization of Abortion,@MetalheadMonty @tom_six I followed him before...,NONE,NONE
1076,Legalization of Abortion,"For he who avenges blood remembers, he does no...",AGAINST,AGAINST
1077,Legalization of Abortion,Life is sacred on all levels. Abortion does no...,AGAINST,AGAINST
1078,Legalization of Abortion,"@ravensymone U refer to ""WE"" which =""YOU"" & a ...",AGAINST,AGAINST


In [119]:
results["correct"] = results["actual"] == results["predicted"]
results.groupby("target")["correct"].mean().sort_values()
# Antheism, Abortion은 그나마 잘 맞지만 나머지는 거의 안 맞음
# Feminist Movement는 현저히 떨어지는 수준

,correct
target,
Feminist Movement,0.498246
Hillary Clinton,0.600000
Legalization of Abortion,0.685714
Atheism,0.722727


In [120]:
proba = model.predict_proba(X_test)
confidence = proba.max(axis=1)  # 열 축을 따라서 max 찾기

In [121]:
results["confidence"] = confidence

In [122]:
results.sort_values("confidence", ascending = False).head(20)
#Antheism이 많음

,target,tweet,actual,predicted,correct,confidence
17,Atheism,#God > My fears God > My insecurities God > My...,AGAINST,AGAINST,True,0.890230
896,Legalization of Abortion,There should be no law or funding which justif...,AGAINST,AGAINST,True,0.883891
149,Atheism,Existing is something god have never been prof...,FAVOR,AGAINST,False,0.880899
61,Atheism,"James 1:14 (#KJV) But every man is TEMPTED, wh...",AGAINST,AGAINST,True,0.876690
210,Atheism,"8 Holy Mary Mother of God, pray for us sinners...",AGAINST,AGAINST,True,0.875287
63,Atheism,"RT @ACatholicPrayer: Holy Trinity, one God, Ha...",AGAINST,AGAINST,True,0.867086
784,Hillary Clinton,So @HillaryClinton received a Subpoena to Test...,AGAINST,AGAINST,True,0.861624
400,Feminist Movement,The Hypocrisy of Feminism is integral part of ...,AGAINST,AGAINST,True,0.857885
1047,Legalization of Abortion,"Lord Jesus,Touch my life with your saving powe...",AGAINST,AGAINST,True,0.856949
432,Feminist Movement,Feminists grades in science class usually are ...,AGAINST,AGAINST,True,0.853464


In [123]:
results.groupby("target")[["confidence", "correct"]].mean()

,confidence,correct
target,,
Atheism,0.612611,0.722727
Feminist Movement,0.524029,0.498246
Hillary Clinton,0.596951,0.600000
Legalization of Abortion,0.551340,0.685714


In [124]:
errors = results[results["correct"]== False]
errors.head()

,target,tweet,actual,predicted,correct,confidence
6,Atheism,"Morality is not derived from religion, it prec...",AGAINST,FAVOR,False,0.388184
13,Atheism,"These days, the cool kids are atheists. #free...",AGAINST,FAVOR,False,0.554904
46,Atheism,"Proud to be #TeamJinnah :) No Bhutto, no Imran...",AGAINST,NONE,False,0.456913
50,Atheism,RT @br_holden: Religions are perfectly happy t...,AGAINST,FAVOR,False,0.457739
52,Atheism,RT @br_holden: Just say no to superstitious th...,AGAINST,FAVOR,False,0.439391


In [125]:
results.groupby(["target"]).size()

,0
target,
Atheism,220
Feminist Movement,285
Hillary Clinton,295
Legalization of Abortion,280


In [126]:
errors.groupby(["actual", "predicted"]).size()

actual   predicted
AGAINST  FAVOR        102
         NONE          26
FAVOR    AGAINST      117
         NONE           8
NONE     AGAINST      137
         FAVOR         20
dtype: int64

In [127]:
errors.groupby(["target", "actual", "predicted"]).size().reset_index(
    name = "count"
).sort_values(
    "count",
    ascending = False
)

# against는 이미 많으므로, actual이 다른 것을 잘못 판단한것이 유효해보임

,target,actual,predicted,count
6,Feminist Movement,AGAINST,FAVOR,78
15,Hillary Clinton,NONE,AGAINST,66
13,Hillary Clinton,FAVOR,AGAINST,41
2,Atheism,FAVOR,AGAINST,27
19,Legalization of Abortion,FAVOR,AGAINST,27
9,Feminist Movement,NONE,AGAINST,24
21,Legalization of Abortion,NONE,AGAINST,24
4,Atheism,NONE,AGAINST,23
8,Feminist Movement,FAVOR,AGAINST,22
18,Legalization of Abortion,AGAINST,NONE,18


In [128]:
errors[
    (errors["target"] == "Hillary Clinton") & (errors["actual"] == "NONE") & (errors["predicted"] == "AGAINST")
]

,target,tweet,actual,predicted,correct,confidence
543,Hillary Clinton,RT @hamackey: RT @SlyDude3677: @msnbc Republi...,NONE,AGAINST,False,0.692652
551,Hillary Clinton,RT @CattyHipster: Live today like it's your la...,NONE,AGAINST,False,0.498570
552,Hillary Clinton,@TheJackOBrien @instapundit BOOM! I believe th...,NONE,AGAINST,False,0.643438
554,Hillary Clinton,"AKA Depends Day ""It all depends"" and Owebama's...",NONE,AGAINST,False,0.487368
556,Hillary Clinton,@WashTimes This is indefensible! One of the du...,NONE,AGAINST,False,0.564315
...,...,...,...,...,...,...
786,Hillary Clinton,@Braidleigh 2016 is the due year for possible ...,NONE,AGAINST,False,0.491992
788,Hillary Clinton,@BadgerGeno @kreichert27 @jackbahlman Too busy...,NONE,AGAINST,False,0.500830
792,Hillary Clinton,"The blacks cant admit that O is a failure. No,...",NONE,AGAINST,False,0.441949
796,Hillary Clinton,@mataharikrishna I'm loving it too! Draw that ...,NONE,AGAINST,False,0.543614


In [129]:
errors.loc[543]["tweet"]
    # 별 내용 없음 / 그냥 RT
    # Republican 을 강하게 탐지한것일지?

'RT @hamackey: RT @SlyDude3677: @msnbc  Republican Menu...pic.twitter.com/z4PH6VTJFl  #SemST'

In [130]:
errors.loc[554]["tweet"]

'AKA Depends Day "It all depends" and Owebama\'s positions make you fill your Depends. @jjauthor @weknowwhatsbest #SemST'

In [131]:
results.loc[[543, 554]]

,target,tweet,actual,predicted,correct,confidence
543,Hillary Clinton,RT @hamackey: RT @SlyDude3677: @msnbc Republi...,NONE,AGAINST,False,0.692652
554,Hillary Clinton,"AKA Depends Day ""It all depends"" and Owebama's...",NONE,AGAINST,False,0.487368


# ■ coef 보기

In [132]:
vectorizer = model.named_steps["tfidf"]
clf = model.named_steps["clf"]

feature_names = vectorizer.get_feature_names_out()
clf.classes_

array(['AGAINST', 'FAVOR', 'NONE'], dtype=object)

In [133]:
class_index = list(clf.classes_).index("FAVOR")

favor_coef = clf.coef_[class_index]

favor_words = pd.DataFrame({
    "feature": feature_names,
    "coef": favor_coef
}).sort_values("coef", ascending=False)

favor_words.head(20)

,feature,coef
9885,women,2.540991
7719,hillaryclinton,1.812189
405,body,1.567880
9877,woman,1.448010
8835,religion,1.352902
1487,choice,1.350569
5898,freethinker,1.243507
8877,right,1.056548
5284,feminist,0.996928
5903,freethinker semst,0.956988


In [134]:
class_index2 = list(clf.classes_).index("AGAINST")

against_coef = clf.coef_[class_index2]

against_words = pd.DataFrame({
    "feature": feature_names,
    "coef": against_coef
}).sort_values("coef", ascending=False)

against_words.head(20)

# feminist는 favor가 강하고 feminists / feminism은 against가 강한것이 독특
# I am a feminist (favor) / feminists are~ (against) 가 많은가?

,feature,coef
5359,feminists,2.256719
9689,unborn,1.766874
6495,god,1.633786
5230,feminism,1.474908
6114,gamergate,1.343314
8704,prolifeyouth,1.241381
8356,murder,1.189367
266,babies,1.152642
7902,jesus,1.111017
6122,gamergate semst,1.050665


In [135]:
class_index3 = list(clf.classes_).index("NONE")

none_coef = clf.coef_[class_index3]

none_words = pd.DataFrame({
    "feature": feature_names,
    "coef": none_coef
}).sort_values("coef", ascending=False)

none_words.head(20)
# 딱히 특징적인 것이 없음. coef 자체도 낮은 편인 것 같음

,feature,coef
274,baltimoreriots,1.033148
9619,today,0.951798
1796,clinton sep,0.949458
8026,legalization abortion,0.851026
96,abortion sep,0.851026
8025,legalization,0.836747
6169,gay,0.796420
9032,sep,0.790963
8998,scotusmarriage,0.777683
9372,socialism,0.743058


In [136]:
X_train_tfidf = vectorizer.fit_transform(X_train)

In [137]:
against_mask = (y_train.values == "AGAINST")
favor_mask = (y_train.values == "FAVOR")
none_mask = (y_train.values == "NONE")

In [138]:
against_tfidf = X_train_tfidf[against_mask]
favor_tfidf = X_train_tfidf[favor_mask]
none_tfidf = X_train_tfidf[none_mask]

In [139]:
against_mean = against_tfidf.mean(axis=0)
favor_mean = favor_tfidf.mean(axis=0)
none_mean = none_tfidf.mean(axis=0)

In [140]:
against_mean = np.asarray(
    against_mean
).flatten()

In [141]:
favor_mean = np.asarray(
    favor_mean
).flatten()

In [142]:
none_mean = np.asarray(
    none_mean
).flatten()

In [143]:
feature_names = vectorizer.get_feature_names_out()

against_words = pd.DataFrame({
    "word": feature_names,
    "mean_tfidf": against_mean
})

against_words.sort_values("mean_tfidf", ascending = False).head(20)

# sep나 타겟명 제외하고 hillary/feminism 관련이 많아보임

,word,mean_tfidf
9021,semst,0.043990
9032,sep,0.043811
7682,hillary,0.037798
1758,clinton,0.032292
83,abortion,0.032174
7696,hillary clinton,0.031059
1796,clinton sep,0.029904
5284,feminist,0.027296
8025,legalization,0.026060
8026,legalization abortion,0.025991


In [144]:
favor_words = pd.DataFrame({
    "word": feature_names,
    "mean_tfidf": favor_mean
})

favor_words.sort_values("mean_tfidf", ascending = False).head(20)

,word,mean_tfidf
5284,feminist,0.046352
9032,sep,0.043547
9021,semst,0.043364
8343,movement sep,0.040415
5323,feminist movement,0.040415
8341,movement,0.040324
7682,hillary,0.028848
83,abortion,0.027283
9885,women,0.026617
1758,clinton,0.024343


In [145]:
none_words = pd.DataFrame({
    "word" : feature_names,
    "mean_tfidf" : none_mean
})

none_words.sort_values("mean_tfidf", ascending = False).head(20)

,word,mean_tfidf
9032,sep,0.046999
9021,semst,0.046400
83,abortion,0.033835
8026,legalization abortion,0.033315
96,abortion sep,0.033315
8025,legalization,0.033290
1758,clinton,0.031777
7682,hillary,0.031655
7696,hillary clinton,0.031561
1796,clinton sep,0.031561


# ■ Tweet만 활용하는 모델

In [146]:
X_train_tweet = train_df["Tweet"]
X_test_tweet = test_df["Tweet"]

In [147]:
model_tweet = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features = 10000,
        ngram_range = (1,2),
        stop_words = "english"
    )),
    ("clf", LogisticRegression(
        max_iter = 1000
    ))
])

In [148]:
model_tweet.fit(X_train_tweet, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf', LogisticRegression(max_iter=1000))])

In [149]:
y_pred_tweet = model_tweet.predict(X_test)

In [150]:
print(classification_report(y_test, y_pred_tweet))
# against recall 이 소폭 올랐지만... none의 recall이 처참해짐

              precision    recall  f1-score   support

     AGAINST       0.67      0.91      0.77       704
       FAVOR       0.39      0.23      0.29       181
        NONE       0.54      0.04      0.07       195

    accuracy                           0.64      1080
   macro avg       0.53      0.39      0.38      1080
weighted avg       0.60      0.64      0.56      1080



In [151]:
# target + tweet 둘다쓴것
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     AGAINST       0.69      0.82      0.75       704
       FAVOR       0.31      0.31      0.31       181
        NONE       0.53      0.19      0.28       195

    accuracy                           0.62      1080
   macro avg       0.51      0.44      0.45      1080
weighted avg       0.60      0.62      0.59      1080



In [152]:
labels = ["AGAINST", "FAVOR", "NONE"]

cm2 = confusion_matrix(y_test, y_pred_tweet, labels=labels)

cm_df2 = pd.DataFrame(
    cm2,
    index=[f"actual_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels]
)

cm_df2
# 대부분을 against로 판단하고, none으로 판단하는 경우는 극히 드물어짐

,pred_AGAINST,pred_FAVOR,pred_NONE
actual_AGAINST,642,58,4
actual_FAVOR,137,42,2
actual_NONE,180,8,7


### Tweet-only vs Target+Tweet

When only the tweet text was used, the model became more biased toward predicting `AGAINST`. Although recall for `AGAINST` increased, performance for `FAVOR` and especially `NONE` dropped substantially.

This suggests that stance detection is not the same as sentiment classification. Without the target, the model appears to rely more on negative wording and general sentiment cues, which can cause many neutral or unrelated tweets to be misclassified as `AGAINST`.

Including the target provides important contextual information because stance is defined as an attitude toward a specific target, not just the emotional tone of the tweet.

# ■ Conclusion

The model achieved moderate performance on stance detection, but several limitations became visible during error analysis.

Including the target together with the tweet improved overall performance substantially. When only tweet text was used, the model became strongly biased toward predicting `AGAINST`, while recall for `NONE` dropped sharply.

This suggests that stance detection is fundamentally different from simple sentiment classification. The stance label depends not only on emotional tone, but also on the relationship between the tweet and a specific target.

The analysis also showed that low-information tweets (such as short retweets or tweets containing only "RT") were especially difficult to classify. In many cases, the model defaulted to dominant or sentiment-related classes when explicit stance signals were weak.

Overall, this notebook highlights some important challenges in NLP stance detection:
- target dependency
- ambiguous language
- class imbalance
- low-context tweets
- shortcut learning based on topic-related signals